# Deteksi Fault 4 Sensor Kelembaban Tanah: EDM-Fuzzy vs JSD-Fuzzy + ANN-LM

**Notebook sidang interaktif.** Semua angka dibaca dari hasil penelitian yang sudah
diekspor; tidak ada model yang dilatih ulang di sini, jadi tiap sel selesai dalam
hitungan detik.

Cara pakai: **Run All** sekali, lalu ubah pilihan di panel kontrol (hari, sensor,
jenis fault, intensitas). Grafik dan tebakan model ikut berubah.

In [ ]:
# === Setup 1/3: pustaka angka ===
import json, sys, time
from pathlib import Path
import numpy as np, pandas as pd

In [ ]:
# === Setup 2/3: grafik dan widget ===
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display, HTML, Markdown

In [ ]:
# === Setup 3/3: lokasi file, mode presentasi ===

ROOT = next(p for p in [Path(".."), Path("."), Path("/kaggle/input/sidang-bahan")]
            if (p / "sidang" / "assets" / "cache_interaktif.npz").exists())
sys.path.insert(0, str((ROOT / "models").resolve()))
C = np.load(ROOT / "sidang" / "assets" / "cache_interaktif.npz")
pd.set_option("display.precision", 3)
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})

_gaya = w.Output()
def _mode(presentasi):
    with _gaya:
        _gaya.clear_output()
        if presentasi:
            display(HTML("<style>.catatan-internal{display:none!important}</style>"))
tombol_mode = w.ToggleButton(value=False, description="Mode presentasi", icon="eye-slash",
                             tooltip="Sembunyikan semua catatan internal")
tombol_mode.observe(lambda ch: _mode(ch["new"]), "value")
display(w.HBox([tombol_mode, _gaya]))
print("Folder kerja:", ROOT.resolve())

<details class="catatan-internal">
<summary><b>Catatan internal — cara memakai notebook ini</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan untuk mahasiswa baru

- **Mode presentasi**: tekan tombol di atas. Semua kotak "Catatan internal" hilang
  dari layar. Kalau tombol tidak bekerja (mis. di VS Code atau tampilan statis
  Kaggle), buka salinan `04_Sidang_Interaktif_presentasi.ipynb` yang memang tidak
  berisi catatan internal.
- **Alur bicara** (±15 menit): masalah → data → fault → entropi (EDM vs JSD) →
  ANN-LM → tebakan model → hasil → statistik → kelemahan → penutup.
- **Aturan menjawab**: angka hanya boleh disebut kalau tampil di notebook atau
  berstatus COCOK di `sidang/qa/01_klaim_bukti.csv`. Kalau ragu: "saya tunjukkan
  di notebook", lalu buka bagian yang relevan.

#### Tanya-jawab penguji

**T1. Bagaimana memakai notebook ini saat sidang?**
- *30 detik:* "Notebook ini memuat model yang sudah dilatih dan fitur yang
  sudah dihitung, jadi setiap sel selesai di bawah dua detik. Bapak/Ibu bisa
  memilih hari, sensor, jenis gangguan, dan intensitasnya, lalu melihat sinyal,
  kurva entropi, dan tebakan model berubah."
- *Untuk mahasiswa baru:* melatih ulang model butuh jam (NB11 di Kaggle
  ±10 jam). Saat presentasi kita tidak melatih; kita **memuat** bobot jaringan
  yang sudah tersimpan, lalu hanya melakukan perkalian matriks (forward pass).
  Itu seperti memakai kalkulator yang rumusnya sudah jadi.
- *Bukti:* `sidang/bukti_demo/04_Sidang_Interaktif.waktu.csv` (13 sel kode,
  total 9,7 s, sel terlama 1,7 s); `sidang/qa/test_notebook_sidang.py`
  (540 kombinasi panel tanpa galat, tiap panggilan < 1 s).

**T2. Apakah model di notebook ini sama dengan model penelitian?**
- *30 detik:* "Sama persis. Bobotnya diekspor dari model tersimpan N = 700 dan
  prediksinya dibandingkan dengan model asli: selisih maksimum nol."
- *Untuk mahasiswa baru:* model asli disimpan sebagai objek Python yang butuh
  scikit-learn (lambat diimpor, ±3–6 detik). Kita ambil angka-angkanya saja
  (bobot W, bias b, rata-rata dan skala normalisasi) ke file `.npz`, lalu
  menulis ulang forward pass dengan numpy. Supaya yakin tidak ada salah salin,
  keluaran kedua versi dibandingkan pada semua jendela uji.
- *Bukti:* `sidang/qa/buat_cache_interaktif.py` fungsi `ekspor_model`
  (assert selisih < 1e-9, terukur 0,00e+00); uji
  `test_bobot_numpy_sama_dengan_model_tersimpan`.

**T3. Kenapa catatan ini tidak tampil saat presentasi?**
- *30 detik:* "Catatan internal hanya untuk persiapan saya. Ada tombol mode
  presentasi dan ada salinan presentasi yang sel catatannya dibuang otomatis."
- *Bukti:* `sidang/qa/ekspor_notebook_sidang.py` (gagal kalau masih ada sel
  bertanda `catatan-internal`); `sidang/04_Sidang_Interaktif_presentasi.ipynb`.

#### Bukti

- Gerbang angka: `./venv/bin/python sidang/qa/cek_klaim.py` → `GERBANG KLAIM: LULUS`.
- Gerbang notebook: `sidang/qa/jalankan_notebook.py` (waktu tiap sel dicatat).

</details>

## 1. Masalahnya

Sensor kelembaban tanah di sistem irigasi bisa rusak diam-diam: nilainya bergeser,
melonjak, merayap naik, atau macet. Sistem yang percaya pada sensor rusak akan
menyiram terlalu banyak atau terlalu sedikit.

**Pertanyaan penelitian:** kalau sinyal 4 sensor dipotong per jendela waktu dan
tiap sensor diringkas dengan **entropi multiskala**, apakah mengganti ukuran
kemiripan **Euclidean (EDM-Fuzzy)** dengan **Jensen–Shannon Divergence
(JSD-Fuzzy)** membuat ANN-LM lebih baik mengenali (1) ada tidaknya fault,
(2) jenis/kombinasi fault, dan (3) sensor mana yang rusak?

<details class="catatan-internal">
<summary><b>Catatan internal — masalah dan kontribusi</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan untuk mahasiswa baru

Bayangkan empat termometer di satu ruangan. Kalau satu menunjukkan angka aneh,
kita curiga termometer itu rusak karena tiga lainnya "normal". Penelitian ini
melakukan hal yang sama untuk sensor kelembaban tanah, tetapi tidak membandingkan
angkanya langsung. Yang dibandingkan adalah **pola** sinyalnya: seberapa teratur
atau acak naik-turunnya. Ukuran keteraturan itu disebut **entropi**.

Entropi dihitung dengan cara membandingkan potongan-potongan kecil sinyal: "apakah
potongan ini mirip potongan itu?". Cara mengukur "mirip" itulah yang diganti:
- **EDM** (Euclidean): jarak garis lurus antar angka, seperti penggaris.
- **JSD**: tiap potongan diubah jadi histogram kecil, lalu dibandingkan bentuk
  histogramnya.

Kontribusinya: (1) JSD sebagai ukuran kemiripan di dalam fuzzy entropy multiskala,
(2) evaluasi berantai sampai identifikasi sensor, (3) pelaporan jujur kapan
keunggulannya signifikan dan kapan tidak. Rumusan kebaruan yang aman ada di
`sidang/qa/07_pustaka_dan_kebaruan.md`.

#### Tanya-jawab penguji

**T4. Penelitian ini tentang apa? (Q1)**
- *30 detik:* "Mendeteksi gangguan pada empat sensor kelembaban tanah dari
  pola sinyalnya. Sinyal dipotong menjadi jendela, tiap sensor diringkas dengan
  entropi multiskala, lalu jaringan saraf yang dilatih Levenberg–Marquardt
  menebak jenis gangguan dan sensor yang rusak. Yang saya uji: mengganti ukuran
  kemiripan Euclidean (EDM-Fuzzy) dengan Jensen–Shannon (JSD-Fuzzy)."
- *Untuk mahasiswa baru:* bayangkan empat termometer di satu kebun. Kalau satu
  mulai "melantur", angka rata-ratanya mungkin tetap wajar, tetapi *polanya*
  berubah: terlalu rata (macet), tiba-tiba melonjak (spike), atau pelan-pelan
  bergeser (drift). Entropi adalah angka yang mengukur seberapa "rumit" pola
  itu. Jaringan saraf belajar dari 60 angka entropi (4 sensor × 15 skala) untuk
  memutuskan "normal atau rusak, dan rusak jenis apa".
- *Bukti:* NB11 §0; `deteksi_fault_core.py` (pipeline lengkap).

**T5. Apa kebaruannya? (Q5)**
- *30 detik:* "Tempat JSD dipasang: sebagai jarak antar pasangan template di
  dalam fuzzy entropy. Penelusuran pustaka tidak menemukan karya yang sama;
  yang ada memakai jarak geometris di dalam entropi, atau memakai JSD pada level
  seluruh sinyal."
- *Untuk mahasiswa baru:* fuzzy entropy membandingkan setiap potongan pendek
  sinyal (template) dengan potongan lain. Biasanya "beda" diukur dengan
  penggaris (jarak Euclidean). Di sini setiap potongan diubah dulu menjadi
  histogram kecil, lalu dua histogram dibandingkan dengan JSD — ukuran beda dua
  sebaran peluang.
- *Bukti:* `07_pustaka_dan_kebaruan.md` (b) tabel prior-art dan log pencarian
  (f); sumber baseline Zhou dkk. 2021, IEEE TII, doi:10.1109/TII.2020.3009139.

**T6. EDM-Fuzzy itu buatan Anda?**
- *30 detik:* "Bukan. EDM-Fuzzy dari Zhou dkk. 2021. Saya memakainya sebagai
  pembanding dan mengganti hanya ukuran jaraknya."
- *Bukti:* `pustaka.bib` kunci `zhou2021edmfuzzy`.

**T7. Kenapa empat sensor, bukan satu, dan kenapa tidak dirata-rata? (Q2, Q13)**
- *30 detik:* "Rata-rata menyembunyikan sensor mana yang rusak dan menurunkan
  skor: F1 0,821 untuk empat kanal terpisah vs 0,748 setelah dirata-rata."
- *Untuk mahasiswa baru:* kalau sensor A naik 10 dan sensor B turun 10,
  rata-ratanya tidak berubah — gangguannya hilang dari pandangan.
- *Bukti:* NB02.

#### Bukti

- `Bahan_Sidang.md` Q1–Q5; `sidang/qa/07_pustaka_dan_kebaruan.md` (prior-art).

</details>

## 2. Data: 4 sensor, ±98 hari

Sensor mencatat tiap 30 detik; broker menyinkronkan keempatnya lalu merata-rata
per 5 menit. Grafik di bawah: rata-rata per jam, seluruh periode.

In [ ]:
jam = C["ringkas_per_jam"]
hari = np.arange(len(jam)) / 24
fig, ax = plt.subplots(figsize=(11, 3))
for j in range(4):
    ax.plot(hari, jam[:, j], lw=0.8, label=f"S{j+1}")
for h0 in C["hari"]:
    ax.axvspan(h0, h0 + int(C["win"]) * 300 / 86400, color="0.85", zorder=0)
ax.set_xlabel("hari sejak awal rekaman"); ax.set_ylabel("kelembaban (%)")
ax.set_title("Data lapangan (abu-abu = tiga jendela yang dipakai di panel interaktif)")
ax.legend(ncol=4, fontsize=8); plt.tight_layout(); plt.show()
print(f"{int(C['n_sampel_5menit']):,} titik 5 menit = {int(C['n_sampel_5menit']) * 300 / 86400:.1f} hari")

<details class="catatan-internal">
<summary><b>Catatan internal — data, sinkronisasi, nilai nol S4</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan untuk mahasiswa baru

- **Broker** = penampung yang menerima data keempat sensor dan menyusunnya di satu
  sumbu waktu. Sensor tidak dirata-rata menjadi satu karena identitas tiap sensor
  diperlukan untuk menunjuk sensor yang rusak (notebook 02).
- **5 menit**: 10 titik 30 detik dirata-rata jadi satu. Derau berkurang, pola
  harian tetap.
- **Tidak ada baris dibuang karena nilainya aneh.** Nilai macet dan nol di S4
  dilaporkan, bukan dibuang, karena justru bisa gejala fault.
- Bagian grafik yang lurus sempurna di keempat sensor ada di file data asli,
  bukan hasil pengolahan.

#### Tanya-jawab penguji

**T8. Data dari mana, berapa banyak? (Q6, Q7)**
- *30 detik:* "Empat sensor kelembaban tanah, 14 September–21 Desember 2025,
  sampel tiap 30 detik, 281.721 baris. Saya rata-ratakan per 5 menit menjadi
  28.173 baris, 97,8 hari."
- *Untuk mahasiswa baru:* 30 detik → 5 menit artinya setiap 10 angka diganti
  rata-ratanya. Contoh: 40, 41, 40, 42, 41, 40, 41, 40, 42, 41 → 40,8. Derau
  kecil teredam, pola harian tetap.
- *Bukti:* NB11 §1–§2; `data_sensor.csv`.

**T9. Nilai nol di S4 diapakan? (Q9)**
- *30 detik:* "Tidak dibuang, dilabeli normal, dan dilaporkan sebagai batasan:
  11.371 sampel (4,0 %), tiga blok macet di 16–21 Desember plus 50 sampel
  pendek."
- *Untuk mahasiswa baru:* nol masih di rentang fisik 0–100 %, jadi tidak bisa
  disebut "salah ketik". Pembacaan macet di nol justru gejala gangguan; kalau
  dibuang, kita menyembunyikan kasus sulit.
- *Bukti:* NB11 §2b; Bahan Q9.

**T10. Missing value dan outlier? Sinkronisasi waktu? (Q8, Q14, Q15)**
- *30 detik:* "Tidak ada NaN dan tidak ada nilai di luar 0–100 %. Stempel waktu
  diubah ke UTC, diurutkan, duplikat dibuang, lalu dipaksa ke grid 5 menit;
  slot bolong nol."
- *Bukti:* NB11 §2 sel 7, §2b sel 8.

**T11. "Broker" itu apa? (Q12)**
- *30 detik:* "Komponen yang menyatukan empat aliran sensor menjadi satu tabel
  berwaktu sama tanpa melebur identitas sensornya; bukan message broker
  produksi."
- *Bukti:* NB02 sel 1–13; `deteksi_fault_core.sync_broker`.

#### Bukti

- Ledger: B-09a…g (nilai nol S4), R-ILU1…5 (mutu data), P-08 (segmen lurus).
- Notebook: `Ilustrasi_Data_Akuisisi_dan_Fault.ipynb` §1–§2, `07` §3b.

</details>

In [ ]:
# === Panel kontrol (dipakai bagian 3, 4, dan 6) ===
pil_hari = w.Dropdown(options=[(f"hari {int(h)}", i) for i, h in enumerate(C["hari"])], description="Jendela")
pil_sensor = w.Dropdown(options=[(f"S{s+1}", s) for s in range(4)], value=1, description="Sensor rusak")
pil_fault = w.Dropdown(options=[(f, i) for i, f in enumerate(C["fault"])], value=4, description="Fault")
pil_int = w.Dropdown(options=[(f"×{int(k)}" + (" (penelitian)" if k == 1 else ""), i)
                              for i, k in enumerate(C["intensitas"])], description="Intensitas")
PANEL = w.HBox([pil_hari, pil_sensor, pil_fault, pil_int])
KONTROL = dict(hi=pil_hari, s=pil_sensor, fi=pil_fault, ki=pil_int)

def hidupkan(fn, kontrol=None, panel=None):
    # Tampilkan hasil fn(**pilihan) sebagai output sel biasa (terlihat juga di tampilan
    # statis) lalu perbarui di tempat setiap kali pilihan panel berubah.
    kontrol = kontrol or KONTROL
    display(panel or PANEL)
    pegangan = []
    def render(*_):
        isi = fn(**{k: v.value for k, v in kontrol.items()})
        if not pegangan:
            pegangan.extend(display(x, display_id=True) for x in isi)
        else:
            for h, x in zip(pegangan, isi):
                h.update(x)
        plt.close("all")
    for v in kontrol.values():
        v.observe(render, "value")
    render()

display(PANEL)

## 3. Fault disuntikkan ke data asli

Data lapangan dianggap **bersih**; fault dibuat dengan simulator lalu disuntikkan
ke satu sensor. Empat jenis: **drift** (merayap naik pelan), **spike** (lonjakan
sesaat), **bias** (bergeser tetap), **hardware** (macet/nilai hilang); "semua" =
keempatnya sekaligus.

In [ ]:
_t_jam = np.arange(int(C["win"])) * 300 / 3600
FIG_F, AX_F = plt.subplots(1, 2, figsize=(11, 2.8), gridspec_kw={"width_ratios": [3, 1]})
L_asli, = AX_F[0].plot(_t_jam, _t_jam * 0, color="tab:blue", lw=1, label="asli")
L_rusak, = AX_F[0].plot(_t_jam, _t_jam * 0, color="tab:red", lw=0.9, alpha=0.8, label="diberi fault")
L_beda, = AX_F[1].plot(_t_jam, _t_jam * 0, color="tab:red", lw=0.8)
AX_F[0].set_xlabel("jam dalam jendela (N = 700 = 58 jam)"); AX_F[0].legend(fontsize=8)
AX_F[1].set_title("selisih (fault − asli)", fontsize=9); AX_F[1].set_xlabel("jam")
FIG_F.tight_layout(); plt.close(FIG_F)

def gambar_fault(hi, s, fi, ki):
    # satu figur dipakai ulang; hanya data garis yang diganti (respons cepat)
    bersih, rusak = C["sig_bersih"][hi][:, s], C["sig_fault"][fi, ki, s, hi]
    L_asli.set_ydata(bersih); L_rusak.set_ydata(rusak); L_beda.set_ydata(rusak - bersih)
    AX_F[0].set_ylabel(f"S{s+1} (%)")
    for ax in AX_F:
        ax.relim(); ax.autoscale_view()
    return [FIG_F, Markdown(f"Porsi sampel S{s+1} yang kena fault di jendela ini: "
                            f"**{C['porsi'][fi, ki, s, hi]:.0%}**")]

hidupkan(gambar_fault)

<details class="catatan-internal">
<summary><b>Catatan internal — simulator fault</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan untuk mahasiswa baru

- Fault **disimulasikan** karena data lapangan tidak punya label "rusak" yang
  pasti. Dengan simulasi kita tahu persis kapan dan di sensor mana fault terjadi,
  jadi jawaban benarnya diketahui.
- Rumus singkat (lihat `deteksi_fault_core.py` §3):
  drift = tambah laju × waktu, hanya di 3 episode 3 hari lalu di-reset;
  spike = lonjakan berkala setinggi 0,08 × SD; bias = + 0,08 poin tetap;
  hardware = ±3 % sampel diganti nilai dari waktu lain (macet) dan 5 % hilang
  (NaN, lalu diisi nilai sebelumnya). Parameter tertulis 8 %, tetapi sampel yang
  hilang menimpa sebagian sampel macet (kelemahan F4).
- Intensitas ×10 dan ×50 di panel hanya untuk **melihat**; model dilatih dengan ×1.
  Uji intensitas yang sebenarnya ada di notebook 13.
- Pertanyaan jebakan: "bias 0,08 poin itu kecil sekali?" — ya, dan itu sebabnya
  bias hampir tidak terdeteksi (lihat bagian 4 dan kelemahan B2).

#### Tanya-jawab penguji

**T12. Kenapa gangguannya disimulasikan? (Q10)**
- *30 detik:* "Data asli tidak berlabel rusak dan saya tidak boleh merusak
  sensor. Simulasi memberi label pasti dan bisa diulang dengan seed tetap.
  Batasnya: hasil berlaku untuk gangguan yang dimodelkan."
- *Untuk mahasiswa baru:* ini seperti latihan pemadam kebakaran dengan api
  buatan: kita tahu persis di mana apinya, jadi bisa menilai apakah petugas
  menemukannya.
- *Bukti:* NB11 §3; `02_kelemahan.md` B (validitas konstruk).

**T13. Tulis model tiap gangguan. (Q17, P17)**
- *30 detik:* "Drift: tambahan yang naik linear 1 poin/hari selama episode
  3 hari. Spike: lonjakan sesaat dengan peluang 1,5 %. Bias: geser tetap.
  Hardware: sebagian titik macet atau hilang."
- *Untuk mahasiswa baru:* sinyal 40, 40, 40, 40.
  Bias +2 → 42, 42, 42, 42. Spike di titik ke-3 → 40, 40, 47, 40.
  Drift → 40, 40,003, 40,007, 40,010 (1 poin/hari = 0,00347 poin per 5 menit).
  Macet → 40, 41, 41, 41 (nilai terakhir diulang).
- *Bukti:* `deteksi_fault_core.py` `simulate_drift_fault`,
  `simulate_spike_fault`, `simulate_bias_fault`, `simulate_hardware_fault`;
  `04_walkthrough_kode.md` baris 18.

**T14. Kenapa drift hanya 1 poin/hari? (Q11, Q42)**
- *30 detik:* "Agar tidak mustahil secara fisik: versi awal 57,6 poin/hari dan
  menembus 100 % dalam sehari. Rentang 0,003–0,17 poin/hari yang saya pakai
  sebagai pembanding adalah **asumsi skenario**; saya tidak menemukan sumber
  terbit yang mengukur laju drift sensor kelembaban tanah per hari."
- *Untuk mahasiswa baru:* dalam jendela N = 200 (16,7 jam) drift hanya
  menambah ±0,7 poin, lebih kecil dari ayunan harian sinyal. Karena itu drift
  memang paling sulit dideteksi, dan itu dilaporkan.
- *Bukti:* README "Koreksi simulator drift"; `07_pustaka_dan_kebaruan.md` (e).
- *Serangan lanjutan:* "Jadi skenario Anda tidak realistis?" → "Arahnya
  realistis (lambat), besarnya asumsi. Uji intensitas NB13 menunjukkan drift
  baru terdeteksi (AUC 0,80–0,85) pada total 60 poin."

**T15. Hardware malfunction 8 % macet dan 5 % hilang? (F4)**
- *30 detik:* "Total titik rusak 8 %: 5 % hilang (NaN) dan 3 % macet, karena
  titik hilang adalah bagian dari titik macet lalu ditimpa NaN. Deskripsi lama
  saya koreksi."
- *Bukti:* `deteksi_fault_core.py` `simulate_hardware_fault`;
  `02_kelemahan.md` F4.

**T16. Kenapa bias konstan tidak terdeteksi? (Q47)**
- *30 detik:* "Karena setiap template dikurangi rata-ratanya sebelum
  dibandingkan. Geser tetap ikut hilang. AUC bias sekitar 0,50 dari 0,08 sampai
  5 poin."
- *Untuk mahasiswa baru:* template [42, 43] dikurangi rata-rata 42,5 menjadi
  [−0,5, 0,5]; template tanpa bias [40, 41] juga menjadi [−0,5, 0,5]. Entropi
  melihat keduanya identik. Panel fault "bias" di notebook menunjukkan kurva
  entropi hampir tidak bergerak.
- *Bukti:* NB12 `12_sweep_fault_param.csv`; `embed_matrix_centered`.

**T17. Kapan jendela dilabel rusak? (Q22, P18)**
- *30 detik:* "Kalau porsi titik rusak di jendela > 1 %; jendela di bawah itu
  dibuang supaya model tidak belajar label yang salah."
- *Bukti:* `FAULT_RATIO_THR`; `11_akuntansi_window.csv`.

#### Bukti

- Ledger: R13-* (uji intensitas), B-11 (laju drift).
- `sidang/qa/02_kelemahan.md` B1–B3, F4–F5; notebook 13 dan 14.

</details>

## 4. Entropi multiskala: EDM-Fuzzy vs JSD-Fuzzy

Tiap sensor diringkas menjadi **15 angka** (satu per skala τ = 1..15): seberapa
tidak teratur polanya ketika sinyal dilihat makin kasar. Empat sensor × 15 = **60
fitur** per jendela. Garis putus-putus = setelah fault.

In [ ]:
T_SK = C["fit_fault"].shape[-1]; _tau = np.arange(1, T_SK + 1)
FIG_E, AX_E = plt.subplots(1, 2, figsize=(11, 2.8), sharex=True)
L_E = [(ax.plot(_tau, _tau * 0.0, "o-", ms=3, label="asli")[0],
        ax.plot(_tau, _tau * 0.0, "s--", ms=3, label="fault")[0]) for ax in AX_E]
for ax in AX_E:
    ax.set_xlabel("skala τ"); ax.legend(fontsize=8)
AX_E[0].set_ylabel("entropi"); FIG_E.tight_layout(); plt.close(FIG_E)

def gambar_entropi(hi, s, fi, ki):
    beda = []
    for mi, ax in enumerate(AX_E):
        asli = C["fit_bersih"][mi, hi, s * T_SK:(s + 1) * T_SK]
        rusak = C["fit_fault"][mi, fi, ki, s, hi]
        L_E[mi][0].set_ydata(asli); L_E[mi][1].set_ydata(rusak)
        ax.set_title(f"{C['metode'][mi]} — S{s+1}", fontsize=10)
        ax.relim(); ax.autoscale_view()
        beda.append(np.abs(rusak - asli).mean())
    return [FIG_E, Markdown("Rata-rata perubahan entropi akibat fault: " +
                            " · ".join(f"{C['metode'][mi]} **{beda[mi]:.3f}**" for mi in range(2)))]

hidupkan(gambar_entropi)

<details class="catatan-internal">
<summary><b>Catatan internal — fuzzy entropy, EDM, JSD</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan untuk mahasiswa baru

Langkah fuzzy entropy untuk satu sensor pada satu skala (contoh angka lengkap di
`sidang/qa/03_algoritma_dengan_tangan.md`):
1. **Coarse-graining komposit**: rata-rata tiap τ titik berurutan (skala τ),
   diulang untuk τ titik awal yang berbeda lalu hasil entropinya dirata-rata.
2. **Embedding**: potong jadi vektor pendek panjang m = 2 (dan m + 1 = 3), kurangi
   rata-rata tiap vektor (*centering*). Akibatnya pergeseran tetap (bias) hilang.
3. **Kemiripan**: bandingkan tiap vektor dengan 128 vektor acuan acak, lalu
   μ = 1 / (1 + d / r).
   - EDM: d = jarak Euclidean antar vektor, r = 0,2 × SD sinyal.
   - JSD: tiap vektor diubah jadi histogram lunak 8 bin (kernel Gauss di rentang
     ±3 SD), d = √JSD antar histogram (jarak Jensen–Shannon), r = 0,2 × √ln 2.
4. **Entropi** = ln(rata-rata kemiripan m ÷ rata-rata kemiripan m+1).
   Sinyal teratur → pola panjang tetap mirip → entropi kecil.

Kenapa JSD bisa lebih peka: ia membandingkan **bentuk sebaran** nilai di dalam
potongan, bukan jarak titik ke titik, sehingga lonjakan dan macet mengubah bentuk
histogram walau jaraknya kecil. Harganya: ekstraksi JSD lebih lambat.

#### Tanya-jawab penguji

**T18. Apa itu entropi di sini? (Q3, Q23, P9)**
- *30 detik:* "Fuzzy entropy mengukur seberapa sering pola pendek yang mirip
  tetap mirip ketika diperpanjang satu titik. Sinyal teratur: entropi kecil.
  Sinyal acak: entropi besar."
- *Untuk mahasiswa baru:* rumusnya FE = ln(φ_m / φ_{m+1}). φ_m adalah rata-rata
  "kemiripan" antar semua pasangan template sepanjang m = 2. Kalau pasangan yang
  mirip di panjang 2 tetap mirip di panjang 3, φ hampir sama dan ln-nya dekat 0.
  Contoh angka pada deret x = [3, 5, 4, 6, 5, 7, 6, 4]: r = 0,2 · sd =
  0,261861; jarak Euclidean template [−1, 1] dan [0,5, −0,5] = 2,121320,
  keanggotaan μ = 1/(1 + 2,121320/0,261861) = 0,109879.
- *Bukti:* `03_algoritma_dengan_tangan.md` (b), dicek otomatis oleh
  `cek_algoritma_tangan.py`; Chen dkk. 2007 (`chen2007fuzzyen`).

**T19. Kenapa multiskala τ = 1..15 dan 60 fitur? (Q24, Q28)**
- *30 detik:* "Gangguan berbeda tampak di skala berbeda: spike di skala halus,
  drift di skala kasar. 15 skala × 4 sensor = 60 fitur."
- *Untuk mahasiswa baru:* skala τ = 2 artinya rata-ratakan tiap dua titik:
  [3, 5, 4, 6] → [4, 5]. Komposit: hitung juga dengan geseran awal (mulai dari
  titik kedua) lalu rata-ratakan hasilnya, supaya tidak bergantung pada titik
  awal.
- *Bukti:* `coarse_grain_cmse`; Costa dkk. 2002, Wu dkk. 2013.

**T20. Di mana tepatnya EDM dan JSD berbeda? (Q4, Q26, Q27, P10)**
- *30 detik:* "Hanya di jarak d antar template. EDM: jarak Euclidean. JSD:
  tiap template diubah jadi histogram lunak 8 bin, jarak = akar JSD. Embedding,
  centroid shifting, fungsi keanggotaan, dan rumus entropi identik."
- *Untuk mahasiswa baru:* EDM seperti membandingkan dua foto piksel demi
  piksel. JSD seperti membandingkan dua foto dari histogram warnanya saja.
  Contoh JSD dari hitungan tangan: d = √0,011811 = 0,108678, ambang
  r_JSD = 0,2·√ln2 = 0,166511, sehingga μ = 1/(1 + 0,108678/0,166511) = 0,605078.
- *Bukti:* `deteksi_fault_core.py` `phi_euclidean`, `vectors_to_distributions`,
  `pairwise_jsd`, `phi_jsd`; `03_algoritma_dengan_tangan.md` (c).

**T21. Kenapa r = 0,2·SD dan r_JSD = 0,2·√ln2? (Q25)**
- *30 detik:* "0,2·SD adalah toleransi lazim dari literatur sample/fuzzy
  entropy, diskalakan dengan simpangan baku supaya adil untuk amplitudo
  berbeda. JSD (log natural) terbatas di [0, ln 2], jadi akar JSD maksimum √ln2;
  ambangnya 20 % dari maksimum itu."
- *Bukti:* `JSD_R_RATIO`, `jsd_fuzzy_entropy_1d`; Richman & Moorman 2000.

**T22. Sifat JSD yang penting? (P11)**
- *30 detik:* "Simetris, selalu berhingga, terbatas [0, ln 2], dan akarnya
  metrik. KL tidak simetris dan bisa tak hingga kalau ada bin nol."
- *Bukti:* Lin 1991; Endres & Schindelin 2003.

**T23. Apa kelemahan konseptual JSD-Fuzzy? (F7)**
- *30 detik:* "Histogram tidak peduli urutan. Template naik [−1, 1] dan turun
  [1, −1] menghasilkan histogram sama, jadi JSD = 0 persis, padahal Euclidean
  memberi 2,83. JSD-Fuzzy mengukur kemiripan sebaran nilai, bukan bentuk pola.
  Akibat lain: nilainya bisa negatif pada deret pendek."
- *Untuk mahasiswa baru:* kata "TAS" dan "SAT" punya huruf yang sama. Kalau
  hanya menghitung huruf, keduanya dianggap kembar.
- *Bukti:* `02_kelemahan.md` F7 (dicek 2026-11-20 dengan
  `vectors_to_distributions` + `pairwise_jsd`); `07_pustaka_dan_kebaruan.md`
  catatan jujur.
- *Serangan lanjutan:* "Kalau buta urutan, kenapa JSD menang?" → "Gangguan yang
  diuji mengubah sebaran amplitudo di dalam template (spike, macet); itu yang
  ditangkap JSD. Untuk gangguan yang hanya membalik arah, JSD tidak peka. Ini
  saya tulis sebagai batasan."

**T24. Kenapa n_ref = 128 titik acuan? (P14)**
- *30 detik:* "Semua pasangan berbiaya kuadrat; 128 acuan acak dengan seed
  tetap memberi aproksimasi yang sama untuk kedua metode."
- *Bukti:* `_pick_ref`.

**T25. Kenapa JSD lebih lambat? (P15)**
- *30 detik:* "Tiap template harus diubah menjadi histogram dan JSD butuh
  logaritma per bin; Euclidean cukup selisih kuadrat. Biaya ada di ekstraksi
  fitur; inferensi per jendela setara."
- *Bukti:* NB14 `hasil14` (1,7–6,8× pada ekstraksi fitur).

#### Bukti

- Kode: `deteksi_fault_core.py` `edm_fuzzy_entropy_1d`, `jsd_fuzzy_entropy_1d`, `pairwise_jsd`.
- Pemeriksa contoh tangan: `sidang/qa/cek_algoritma_tangan.py`.
- Ledger H14-42*, H14-45c/d (biaya ekstraksi).

</details>

## 5. ANN dilatih dengan Levenberg–Marquardt (LM)

60 fitur → jaringan saraf kecil (1–2 lapisan tersembunyi) → kelas. Bobot dicari
dengan LM: langkah **(JᵀJ + λ·diag(JᵀJ))·Δ = −Jᵀr**. λ kecil ≈ Gauss–Newton
(cepat dekat minimum), λ besar ≈ turun gradien (hati-hati jauh dari minimum).
Coba geser λ pada contoh mini di bawah (satu neuron, dua bobot).

In [ ]:
rng_toy = np.random.default_rng(0)
x_toy = np.linspace(0, 4, 30); y_toy = 2.0 * np.tanh(0.8 * x_toy) + rng_toy.normal(0, 0.05, 30)

def lm_mini(lam0, adaptif, iters=25):
    th = np.array([0.2, 3.0]); lam = lam0; hist = []
    for _ in range(iters):
        a, b = th; z = np.tanh(b * x_toy); r = a * z - y_toy
        J = np.column_stack([z, a * x_toy * (1 - z**2)])
        A = J.T @ J; g = J.T @ r
        step = np.linalg.solve(A + lam * np.diag(np.diag(A)) + 1e-12 * np.eye(2), -g)
        th_baru = th + step
        r_baru = th_baru[0] * np.tanh(th_baru[1] * x_toy) - y_toy
        if (r_baru**2).sum() < (r**2).sum():
            th = th_baru; lam = lam / 3 if adaptif else lam
        else:
            lam = lam * 2 if adaptif else lam
        hist.append(((th[0] * np.tanh(th[1] * x_toy) - y_toy) ** 2).mean())
    return th, hist

def gambar_lm(log_lambda, adaptif):
    th, hist = lm_mini(10.0 ** log_lambda, adaptif)
    fig, axs = plt.subplots(1, 2, figsize=(11, 2.6))
    axs[0].semilogy(hist, "o-", ms=3); axs[0].set_xlabel("iterasi"); axs[0].set_ylabel("MSE")
    axs[0].set_title(f"λ awal = {10.0 ** log_lambda:g}, {'adaptif' if adaptif else 'tetap'}", fontsize=10)
    axs[1].plot(x_toy, y_toy, "k.", label="data")
    axs[1].plot(x_toy, th[0] * np.tanh(th[1] * x_toy), "r-", label="hasil fit"); axs[1].legend(fontsize=8)
    plt.tight_layout()
    return [fig]

K_LM = dict(log_lambda=w.FloatSlider(value=-2, min=-4, max=3, step=0.5, description="log10 λ"),
            adaptif=w.Checkbox(value=True, description="λ adaptif (seperti penelitian)"))
hidupkan(gambar_lm, K_LM, w.HBox(list(K_LM.values())))

<details class="catatan-internal">
<summary><b>Catatan internal — ANN-LM dan cara evaluasi</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan untuk mahasiswa baru

- **Kenapa LM**: untuk jaringan kecil dan data ratusan sampel, LM memakai
  informasi lengkung (JᵀJ) sehingga butuh jauh lebih sedikit iterasi daripada
  Adam/gradien biasa. Notebook 09 membandingkan LM, Bayesian regularization, GA,
  dan Adam: LM ≈ BR, keduanya jauh di atas GA dan Adam.
- **Implementasi sendiri** dengan Jacobian analitik (turunan dihitung rumus, bukan
  beda-hingga), aturan λ Nielsen. Klasifikasi dilakukan dengan target one-hot +
  MSE, lalu softmax untuk peluang.
- **Evaluasi**: 5 lipatan *StratifiedGroupKFold*; grup = blok waktu, supaya jendela
  dari waktu yang sama tidak ada di data latih dan uji sekaligus. Arsitektur dipilih
  di dalam lipatan latih saja (nested CV).
- Contoh mini di atas: λ tetap kecil kadang melompat terlalu jauh, λ tetap besar
  lambat; λ adaptif menggabungkan keduanya.

#### Tanya-jawab penguji

**T26. ANN-nya seperti apa? (Q29)**
- *30 detik:* "Feed-forward, 60 masukan, satu atau dua lapisan tersembunyi
  dari kisi 8 arsitektur, aktivasi relu/tanh, keluaran softmax. Arsitektur
  dipilih di dalam cross-validation."
- *Bukti:* NB11 §10; `models/meta.json`.

**T27. Tulis rumus update LM dan arti λ. (P1, P2, P3)**
- *30 detik:* "(JᵀJ + λ·diag(JᵀJ))·Δ = −Jᵀr. λ kecil: mendekati Gauss–Newton,
  cepat dekat minimum. λ besar: langkah kecil searah gradien, aman. Galat naik:
  langkah ditolak, λ dikali 2. Galat turun: diterima, λ dikecilkan menurut
  rasio ρ (aturan Nielsen)."
- *Untuk mahasiswa baru:* LM seperti menuruni bukit dalam kabut. Kalau
  langkah besar membuat Anda naik (galat bertambah), Anda mundur dan mencoba
  langkah lebih kecil. Di contoh mainan: iterasi 1 ditolak (ρ = −0,58),
  λ 0,001 → 0,002; pada iterasi 9 galat turun dari 1,453047 ke 0,194302.
- *Bukti:* `03_algoritma_dengan_tangan.md` (a); `lm_fit`; Marquardt 1963,
  Hagan & Menhaj 1994, Madsen dkk. 2004.

**T28. Bagaimana Jacobian analitik dihitung dan dibuktikan? (Q31, P4)**
- *30 detik:* "Backpropagation per keluaran. Dibuktikan dengan membandingkan
  terhadap Jacobian numerik beda hingga."
- *Untuk mahasiswa baru:* turunan numerik: geser satu bobot sedikit (ε), lihat
  berapa keluaran berubah, bagi ε. Kalau rumus analitik dan cara numerik
  memberi angka sama sampai ±10⁻¹⁰, rumusnya benar.
- *Bukti:* NB11 §9a; `03` (d); `cek_algoritma_tangan.py` (Jacobian tangan vs
  kode, selisih 5,6e-17).

**T29. LM itu untuk regresi; klasifikasinya bagaimana? (P5)**
- *30 detik:* "Target one-hot, kuadrat terkecil pada keluaran linear, softmax
  saat prediksi — cara yang lazim di `trainlm` MATLAB. Kelemahannya
  probabilitasnya tidak terkalibrasi."

**T30. Kenapa LM, bukan Adam? (Q30, P7)**
- *30 detik:* "LM metode pada paper acuan dan cocok untuk jaringan kecil. Di
  arsip NB09 LM 0,433, BR 0,429, GA 0,306, Adam 0,268; LM dan BR tidak berbeda,
  keduanya jauh di atas GA dan Adam."
- *Bukti:* NB09 §10.

**T31. Kenapa jaringan dibatasi kecil? (P6)**
- *30 detik:* "JᵀJ berukuran P×P dan harus diselesaikan tiap iterasi, biayanya
  O(P³). Karena itu parameter dibatasi sekitar 5 ribu."

**T32. Jelaskan nested grouped CV. (Q33, P20)**
- *30 detik:* "Luar: StratifiedGroupKFold 5 lipatan, grup = blok waktu. Dalam:
  2 lipatan untuk memilih arsitektur dengan macro-F1. Model terpilih dilatih
  ulang di data latih luar dan dinilai di data uji luar."
- *Untuk mahasiswa baru:* jendela bertetangga berbagi 50 % isi. Kalau satu di
  latih dan tetangganya di uji, model "sudah pernah melihat" separuh soal ujian.
  Mengelompokkan per blok waktu memisahkan soal latihan dan soal ujian.
- *Bukti:* NB11 §10; Roberts dkk. 2017, Bergmeir & Benítez 2012.

**T33. Kenapa macro-F1? (P23)**
- *30 detik:* "Kelas tidak seimbang; macro-F1 memberi bobot sama pada tiap
  kelas, sehingga kelas sulit tidak tertutup kelas mudah."
- *Untuk mahasiswa baru:* 90 jendela normal dan 10 rusak. Model yang selalu
  menjawab "normal" akurasinya 90 %, tetapi F1 kelas rusak 0. Macro-F1 =
  rata-rata (F1 normal ≈ 0,95, F1 rusak 0) ≈ 0,47 — angka itu jujur.

**T34. Bobot awal Xavier? (A5, F2)**
- *30 detik:* "Jujur: urutan `lm_pack`/`lm_unpack` berbeda, sehingga sebagian
  bobot lapisan keluaran mulai dari nol. Konsisten saat latih dan prediksi,
  sama untuk kedua metode. Teks skripsi tidak menyebut 'Xavier dengan bias
  nol'."
- *Bukti:* `02_kelemahan.md` A5, F2.

#### Bukti

- Kode: `deteksi_fault_core.py` `lm_fit`, `lm_residual_jacobian`, `inner_search_best`.
- Ledger R09-* (perbandingan pelatih), B-29 (grid 8 arsitektur).
- `sidang/qa/04_walkthrough_kode.md`, `05_bank_pertanyaan.md` §1.

</details>

## 6. Tebakan model tersimpan untuk jendela pilihan

Model N = 700 yang disimpan (`models/`) dipakai langsung: **S1** menjawab "normal
atau rusak", model skenario sesuai jumlah fault menjawab jenisnya, dan model
sensor-ID menunjuk sensor yang rusak (skor > 0,5).

In [ ]:
# Bobot 10 model N = 700 dari models/ (diekspor ke numpy oleh sidang/qa/buat_cache_interaktif.py,
# prediksinya sudah dicek sama dengan model asli; selisih < 1e-9).
MZ = np.load(ROOT / "sidang" / "assets" / "model_sidang.npz")
META = json.loads(str(MZ["_meta"]))

def maju(kunci, X, sensor=False):
    X = np.array(X, float, copy=True)
    nan = np.isnan(X)
    if nan.any():
        X[nan] = np.broadcast_to(MZ[kunci + "|fill"], X.shape)[nan]
    a = (X - MZ[kunci + "|mean"]) / MZ[kunci + "|scale"]
    n, act = int(MZ[kunci + "|n"]), str(MZ[kunci + "|act"])
    for i in range(n):
        a = a @ MZ[f"{kunci}|W{i}"] + MZ[f"{kunci}|b{i}"]
        if i < n - 1:
            a = np.maximum(a, 0.0) if act == "relu" else np.tanh(a)
    if sensor:
        return a
    e = np.exp(a - a.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

MODELS = {}
for k in sorted({"|".join(x.split("|")[:2]) for x in MZ.files if not x.startswith("_")}):
    sc, met = k.split("|")
    MODELS[(sc, met)] = {"class_names": [str(c) for c in MZ[k + "|kelas"]],
                         "detektor": lambda X, k=k: maju(k + "|det", X),
                         "sensor_id": lambda X, k=k: maju(k + "|sen", X, sensor=True),
                         "cv_f1_nb11": float(MZ[k + "|cv_f1_nb11"])}
print(f"{len(MODELS)} model dimuat | dibuat {META['dibuat']} | commit {META['commit']} | N = {META['window']}")

In [ ]:
def tebak(hi, s, fi, ki):
    T = C["fit_fault"].shape[-1]
    sken = "S5" if C["fault"][fi] == "semua" else "S2"
    baris, sens = [], []
    for mi, met in enumerate(C["metode"]):
        f_asli = C["fit_bersih"][mi, hi].copy()
        f_rusak = f_asli.copy(); f_rusak[s * T:(s + 1) * T] = C["fit_fault"][mi, fi, ki, s, hi]
        X2 = np.vstack([f_asli, f_rusak])
        for sc in ["S1", sken]:
            b = MODELS[(sc, met)]; P = b["detektor"](X2)
            for wi, nama in enumerate(["asli", "diberi fault"]):
                k = int(P[wi].argmax())
                baris.append({"Jendela": nama, "Model": sc, "Metode": met,
                              "Tebakan": b["class_names"][k], "Yakin": f"{P[wi, k]:.0%}"})
        skor = MODELS[(sken, met)]["sensor_id"](f_rusak[None])[0]
        sens.append({"Metode": met, **{f"S{j+1}": round(float(v), 2) for j, v in enumerate(skor)},
                     "Ditebak rusak": ", ".join(f"S{j+1}" for j, v in enumerate(skor) if v > 0.5) or "-",
                     "Benar": f"S{s+1}"})
    benar_s = "drift+spike+bias+HW" if sken == "S5" else str(C["fault"][fi])
    return [Markdown(f"**Jawaban benar:** jendela asli = *fault free*; jendela fault = *faulty* (S1) "
                     f"dan *{benar_s}* ({sken}). Intensitas latih = ×1."),
            pd.DataFrame(baris), Markdown("**Sensor mana yang rusak** (skor > 0,5 = rusak):"),
            pd.DataFrame(sens)]

hidupkan(tebak)

<details class="catatan-internal">
<summary><b>Catatan internal — membaca tebakan model</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan untuk mahasiswa baru

- Satu contoh bisa benar bisa salah. Peluang dekat 50 % artinya model ragu. Yang
  adil adalah angka CV (bagian 7), bukan satu contoh.
- Model dilatih pada **semua** jendela skenario lalu disimpan; akurasi pada data
  latih bukan ukuran generalisasi. Angka resmi = F1 CV notebook 11.
- Model hanya mengenal intensitas ×1. Pada ×10/×50 tebakan bisa aneh karena
  polanya di luar data latih — itu jawaban jujur kalau ditanya.
- Demo cadangan dengan banyak jendela acak: `sidang/03_Demo_Model_Tersimpan.ipynb`
  Langkah 6.

#### Tanya-jawab penguji

**T35. Bagaimana membaca tebakan di panel?**
- *30 detik:* "Tabel pertama: tebakan model N = 700 (S1 normal/rusak dan S2
  atau S5 jenis gangguan) untuk jendela asli dan jendela yang diberi gangguan,
  EDM dan JSD berdampingan, dengan tingkat keyakinan. Tabel kedua: skor tiap
  sensor; skor > 0,5 berarti sensor itu ditebak rusak."
- *Untuk mahasiswa baru:* softmax mengubah skor mentah menjadi peluang yang
  jumlahnya 1. Skor [2, 0] → e²/(e² + e⁰) = 7,39/8,39 = 0,88 untuk kelas
  pertama.
- *Bukti:* `maju` di `buat_cache_interaktif.py`; sel prediksi notebook.

**T36. Kenapa tebakan kadang salah di panel?**
- *30 detik:* "Karena F1 validasi silang skenario ini memang belum sempurna,
  dan panel memakai satu jendela, bukan rata-rata. Contoh salah lebih jujur
  daripada demo yang dipilih-pilih."
- *Bukti:* `|cv_f1_nb11` di `model_sidang.npz`; `11_rq4_performa_ann.csv`.

**T37. Bisa dipakai real-time? (Q43, P32)**
- *30 detik:* "Belum. Satu keputusan N = 700 butuh 58 jam data; cocok untuk
  pemantauan harian. Untuk notifikasi cepat ada NB15 (per sampel, tanpa
  entropi) dengan aturan persistensi."
- *Bukti:* NB15, NB15b.

#### Bukti

- `models/meta.json` (arsitektur, cv_f1_nb11), `sidang/PANDUAN_DEMO.md`.
- Ledger P-04…P-06 (bukti run demo `sidang/bukti_demo/`).

</details>

## 7. Hasil utama (notebook 11, LM asli, 15 sel = 5 skenario × 3 panjang jendela)

Nilai = F1 makro rata-rata 5 lipatan CV. Hijau = JSD lebih tinggi.

In [ ]:
f1 = pd.read_csv(ROOT / "hasil11" / "exports" / "11_rq5_f1_berpasangan.csv")
f1 = f1[pd.to_numeric(f1["N_window"], errors="coerce").notna()].copy()
f1["N"] = f1["N_window"].astype(float).astype(int)
tabel = f1.pivot(index="Scenario", columns="N", values="Difference")
display(Markdown("**Selisih F1 (JSD − EDM)** per sel:"))
display(tabel.style.format("{:+.3f}").background_gradient(cmap="RdYlGn", vmin=-0.1, vmax=0.1))
r = f1[["EDM-Fuzzy F1", "JSD-Fuzzy F1"]].astype(float)
print(f"JSD menang {int((f1['Difference'].astype(float) > 0).sum())} dari {len(f1)} sel | "
      f"rata-rata F1 EDM {r['EDM-Fuzzy F1'].mean():.3f} vs JSD {r['JSD-Fuzzy F1'].mean():.3f}")
sid = pd.read_csv(ROOT / "hasil11" / "exports" / "11_sensor_ringkas.csv")
display(Markdown("**Identifikasi sensor** (dirantai dari prediksi out-of-fold):")); display(sid)

<details class="catatan-internal">
<summary><b>Catatan internal — cara membaca hasil</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan untuk mahasiswa baru

- **F1 makro**: rata-rata F1 semua kelas, sehingga kelas kecil dihitung sama
  penting. Akurasi bisa menipu kalau satu kelas dominan.
- S1 (normal vs rusak) dan S5 (empat fault sekaligus) paling mudah; S2–S4 sulit
  karena kelasnya banyak dan beberapa fault (bias, drift kecil) hampir tak terlihat.
- Kalimat aman: "JSD-Fuzzy unggul **konsisten** (menang di sebagian besar sel),
  tetapi pada protokol utama (NB11) keunggulan gabungannya **belum signifikan**;
  dengan fault bentuk episode (NB14) keunggulannya signifikan."

#### Tanya-jawab penguji

**T38. Hasil utamanya? (Q35, Q36)**
- *30 detik:* "JSD-Fuzzy lebih tinggi di 11 dari 15 sel, rata-rata selisih
  F1 +0,028. Per ukuran jendela: N = 200 menang 3/5, N = 700 5/5,
  N = 1000 3/5."
- *Untuk mahasiswa baru:* satu sel = satu skenario (S1–S5) × satu ukuran
  jendela (200/700/1000). Di tiap sel kedua metode dilatih dengan protokol yang
  sama; kita catat selisih F1-nya.
- *Bukti:* NB11 §10; `11_rq4_selisih.csv`; `statistik_sel.csv` baris NB11.

**T39. Hasil identifikasi sensor? (Q37, F1)**
- *30 detik:* "JSD 0,675 vs EDM 0,634, menang 9 dari 15. Catatan jujur: tahap
  ini memakai pembagian acak 70/30, jadi angka absolutnya optimistis; selisih
  kedua metode tetap adil karena pembagiannya sama."
- *Bukti:* NB11 §12; `11_sensor_ringkas.csv`; `02_kelemahan.md` F1.

**T40. JSD lebih tinggi, EDM lebih apa?**
- *30 detik:* "EDM lebih stabil (56 dari 63 pasangan). Keduanya saya
  tampilkan."
- *Bukti:* Bahan "Kalimat siap pakai" 3.

**T41. NB10, NB11, NB14 — kenapa ada tiga hasil?**
- *30 detik:* "NB10 dan NB11 protokol sama, berbeda anggaran sesi; NB14
  memakai simulator fault 'sedang' dengan posisi acak ber-seed. Arah ketiganya
  sama: JSD lebih tinggi."
- *Bukti:* `statistik_sel.csv` (NB10 11/15, NB11 11/15, NB14 12/15).

#### Bukti

- Ledger R11-01…R11-18, H14-01… (NB14).
- Notebook 11 §10–§12, notebook 14; `hasil11/Ringkasan_untuk_laporan.md`.

</details>

## 8. Seberapa kuat buktinya?

Uji berpasangan per notebook, ditambah ukuran efek (Cohen dz), interval
kepercayaan bootstrap 95 % untuk selisih rata-rata, dan koreksi Holm untuk tiga
panjang jendela.

In [ ]:
st = pd.read_csv(ROOT / "sidang" / "qa" / "hasil_uji" / "statistik_sel.csv")
kol = ["notebook", "N", "menang_JSD", "n_sel", "rerata_selisih", "p_t", "p_wilcoxon",
       "p_t_holm_3N", "cohen_dz", "ci95_bawah", "ci95_atas"]
display(st[kol].style.format({c: "{:.3f}" for c in kol[4:]}, na_rep="—"))
p_lip = ROOT / "sidang" / "qa" / "hasil_uji" / "statistik_lipatan.csv"
if p_lip.exists():
    display(Markdown("**Tingkat lipatan, N = 700** (arsitektur tetap; t terkoreksi Nadeau–Bengio):"))
    display(pd.read_csv(p_lip).round(3))

<details class="catatan-internal">
<summary><b>Catatan internal — statistik dan cara membela angka p</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan untuk mahasiswa baru

- **Uji-t berpasangan**: tiap sel punya dua angka (EDM, JSD) dari data dan lipatan
  yang sama, jadi yang diuji adalah selisihnya.
- **Wilcoxon** tidak mengandaikan selisih berdistribusi normal. Dengan 5 pasangan,
  p terkecil yang mungkin 0,0625, jadi Wilcoxon per-N tidak akan pernah < 0,05.
- **Holm**: tiga uji (N = 200/700/1000) sekaligus memperbesar peluang "kebetulan
  signifikan"; Holm menaikkan p untuk mengoreksinya.
- **Cohen dz** = rata-rata selisih ÷ simpangan baku selisih: ukuran efek yang
  tidak tergantung jumlah sel. ±0,2 kecil, ±0,5 sedang, ±0,8 besar.
- **Bootstrap CI**: ambil ulang 15 sel secara acak 10.000 kali; kalau rentang
  tidak memuat 0, selisihnya konsisten positif. Dengan hanya 15 sel, CI ini
  cenderung sempit; laporkan berdampingan dengan p, bukan pengganti.
- **Nadeau–Bengio**: lipatan CV berbagi data latih, sehingga uji-t biasa terlalu
  optimistis; koreksinya memperbesar variansi dengan faktor (1/k + n_uji/n_latih).

#### Tanya-jawab penguji

**T42. Apakah hasilnya signifikan? (Q39, Q41)**
- *30 detik:* "NB11 gabungan belum: uji-t p = 0,076, Wilcoxon 0,107. Ukuran
  efek sedang (d_z = 0,49), dan interval bootstrap 95 % selisih F1 [+0,002;
  +0,058] tidak memuat nol. Jadi saya menyebutnya konsisten lebih tinggi, bukan
  terbukti. NB14 gabungan signifikan: p = 0,0006, d_z = 1,13, CI [+0,043;
  +0,108]."
- *Untuk mahasiswa baru:* p menjawab "kalau sebenarnya tidak ada beda,
  seberapa sering kita melihat beda sebesar ini karena kebetulan?". p = 0,076
  berarti sekitar 7,6 %, di atas batas 5 %. Bootstrap: ambil ulang 15 selisih
  secara acak dengan pengembalian 10.000 kali, hitung rata-rata tiap kali, lalu
  lihat 2,5 % dan 97,5 % terbawah/teratas.
- *Bukti:* `sidang/qa/hasil_uji/statistik_sel.csv`; `statistik_tambahan.py`.
- *Serangan lanjutan:* "CI tidak memuat nol tapi p > 0,05 — mana yang benar?"
  → "Keduanya pendekatan berbeda pada n = 15: bootstrap persentil cenderung
  sempit pada sampel kecil, uji-t lebih konservatif. Saya ambil yang
  konservatif untuk klaim."

**T43. N = 700 signifikan, kan? (Q41)**
- *30 detik:* "Tanpa koreksi p = 0,027; setelah koreksi Holm untuk tiga ukuran
  jendela p = 0,081. Jadi tidak saya klaim signifikan setelah koreksi."
- *Untuk mahasiswa baru:* menguji tiga hal sekaligus menaikkan peluang satu
  'beruntung'. Holm: urutkan p, kalikan yang terkecil dengan 3, berikutnya
  dengan 2, dan seterusnya (dengan aturan tidak boleh turun).
- *Bukti:* `statistik_sel.csv` kolom `p_t_holm_3N`; Holm 1979.

**T44. Uji-t pada lipatan CV itu bias, kan? (P22, C2)**
- *30 detik:* "Ya, lipatan berbagi data latih sehingga varians diremehkan.
  Saya menambahkan uji-t terkoreksi Nadeau–Bengio. Uji utama saya per sel
  (skenario × N), bukan per lipatan."
- *Untuk mahasiswa baru:* koreksi Nadeau–Bengio memperbesar varians dengan
  faktor (1/k + n_uji/n_latih); hasilnya |t| selalu lebih kecil.
- *Bukti:* `statistik_tambahan.py` `t_nadeau_bengio`, keluaran per lipatan
  `hasil_uji/statistik_lipatan.csv` (dari run purge N = 700); Nadeau & Bengio
  2003; Dietterich 1998; uji `test_nadeau_bengio_*`.

**T45. Kenapa Wilcoxon per N tidak pernah signifikan? (F6)**
- *30 detik:* "Dengan lima pasangan, p eksak dua sisi terkecil 2/2⁵ = 0,0625.
  Jadi secara matematis tidak mungkin < 0,05."

**T46. Ukuran efek? (P25)**
- *30 detik:* "d_z NB11 gabungan 0,49 (sedang), NB14 1,13 (besar)."
- *Bukti:* `statistik_sel.csv` kolom `cohen_dz`; Cohen 1988.

#### Bukti

- `sidang/qa/statistik_tambahan.py` + `test_statistik_tambahan.py` (6 uji, bersumber pustaka).
- `sidang/qa/02_kelemahan.md` C1–C6, F6; ledger R11-03…05.

</details>

## 9. Kelemahan yang kami ketahui (dan apa yang sudah diukur)

In [ ]:
p_ef = ROOT / "sidang" / "qa" / "hasil_uji" / "efek_purge_permutasi.csv"
if p_ef.exists():
    ef = pd.read_csv(p_ef)
    display(Markdown("**Purge** = buang jendela latih yang bertumpang-tindih waktu dengan jendela uji. "
                     "**Permutasi** = label latih diacak; F1 harus turun ke tingkat tebakan."))
    display(ef.round(3))
else:
    print("Hasil uji purge/permutasi belum tersedia.")

| Jenis | Kelemahan | Dampak |
|---|---|---|
| Internal | Jendela bersebelahan berbagi setengah sampel di batas grup CV | skor absolut sedikit optimistis; berlaku sama untuk EDM dan JSD |
| Konstruk | Fault disimulasikan, bukan kerusakan nyata | hasil berlaku untuk bentuk fault yang dimodelkan |
| Konstruk | *Centering* menghapus bias tetap | bias tidak terdeteksi oleh kedua metode |
| Statistik | 15 sel tidak saling bebas; banyak uji | p gabungan dibaca hati-hati, dilengkapi Holm/dz/CI |
| Eksternal | Satu lokasi, 4 sensor, ±98 hari | perlu validasi di lokasi/sensor lain |

<details class="catatan-internal">
<summary><b>Catatan internal — menjawab serangan validitas</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan untuk mahasiswa baru

Pola jawaban tiap kelemahan: **akui → dampak → mitigasi**, dua kalimat. Contoh
untuk kebocoran CV: "Pengelompokan per blok waktu mencegah jendela yang sama bocor,
tetapi jendela di batas blok masih berbagi setengah sampelnya, jadi skor
absolutnya sedikit optimistis. Karena berlaku sama untuk EDM dan JSD, selisihnya
tetap sah, dan kami mengukurnya: lihat tabel purge di atas."

#### Tanya-jawab penguji

**T47. Apakah masih ada kebocoran data? (A1, P21)**
- *30 detik:* "Ada sedikit di batas grup: jendela terakhir satu blok bisa
  tumpang-tindih dengan jendela pertama blok tetangga. Sama untuk kedua metode.
  Saya ukur dengan purge: jendela latih yang tumpang-tindih dengan jendela uji
  dibuang, lalu model N = 700 dilatih ulang."
- *Untuk mahasiswa baru:* purge = membuang soal latihan yang isinya beririsan
  dengan soal ujian. Kalau skor turun banyak setelah purge, berarti skor lama
  terbantu kebocoran.
- *Bukti:* `sidang/qa/uji_purge_permutasi_N700.py`,
  `hasil_uji/efek_purge_permutasi.csv`. Catatan: purge juga mengurangi data
  latih ±20 %, jadi penurunan skor tidak seluruhnya akibat kebocoran.

**T48. Bagaimana tahu model tidak belajar artefak? (kontrol negatif)**
- *30 detik:* "Label saya acak lalu model dilatih ulang dengan protokol sama.
  Kalau pipeline jujur, F1-nya turun ke tingkat tebak-tebakan."
- *Untuk mahasiswa baru:* untuk 2 kelas, tebakan acak memberi F1 sekitar 0,5.
  Kalau model dengan label acak masih jauh di atas itu, ada yang bocor.
- *Bukti:* `hasil_uji/efek_purge_permutasi.csv` varian `permutasi` (S1, S5).

**T49. Validitas eksternal? (D)**
- *30 detik:* "Satu lokasi, satu musim, empat sensor, gangguan disimulasikan.
  Hasil berlaku untuk gangguan yang dimodelkan; langkah berikutnya data lapangan
  yang benar-benar rusak."
- *Bukti:* `02_kelemahan.md` D.

**T50. Data asli atau buatan? (B6, P16)**
- *30 detik:* "Gangguan disuntikkan secara simulasi ke `data_sensor.csv`."
  Asal-usul `data_sensor.csv` **belum dipastikan**: baris awalnya tampak hasil
  interpolasi ke grid 30 detik. Jawab hanya setelah dikonfirmasi ke pembimbing;
  jangan mengarang.
- *Bukti:* `02_kelemahan.md` B6.

**T51. p = 0,0006 di NB14 — apakah karena gangguan diperbesar sampai JSD menang?
(P24, A6)**
- *30 detik:* "Simulator NB14 ditetapkan sebelum membandingkan metode dan sama
  untuk keduanya. Tetap signifikan setelah koreksi ganda."
- *Bukti:* `02_kelemahan.md` A6, C3.

#### Bukti

- `sidang/qa/02_kelemahan.md` (daftar lengkap A1–F7 dengan file:baris).
- `sidang/qa/uji_purge_permutasi_N700.py` → `hasil_uji/purge_permutasi_N700.csv`.

</details>

## 10. Kesimpulan

1. Entropi multiskala 4 sensor + ANN-LM dapat mendeteksi fault dan menunjuk sensor
   yang rusak, paling baik untuk hardware/macet dan kombinasi banyak fault.
2. Mengganti Euclidean dengan **Jensen–Shannon Divergence** memberi keunggulan yang
   konsisten; kekuatan statistiknya bergantung pada bentuk fault (lihat bagian 8).
3. Bias tetap dan drift kecil tetap sulit: keterbatasan ini berasal dari
   *centering* di dalam entropi, bukan dari classifier.

<details class="catatan-internal">
<summary><b>Catatan internal — kalimat penutup dan langkah lanjut</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan untuk mahasiswa baru

Tiga kalimat siap pakai ada di `Bahan_Sidang.md` bagian "Kalimat siap pakai".
Langkah lanjut yang masuk akal: validasi di sensor/lokasi lain, fitur tanpa
centering untuk bias, jendela lebih pendek untuk pemantauan dekat real-time,
dan CV dengan purge sebagai protokol baku.

#### Tanya-jawab penguji

**T52. Kalimat penutup.**
- "Saya mengganti ukuran kemiripan di dalam fuzzy entropy dari Euclidean
  menjadi Jensen–Shannon dan mengujinya sampai identifikasi sensor."
- "JSD-Fuzzy lebih tinggi di 11 dari 15 sel (+0,028); secara gabungan NB11
  belum signifikan (p = 0,076), sehingga saya menyebutnya konsisten, bukan
  terbukti. Pada simulator NB14 selisihnya signifikan (p = 0,0006)."
- "Batasan yang saya laporkan: gangguan disimulasikan, JSD buta urutan,
  sedikit kebocoran di batas grup, dan bias konstan tidak terdeteksi oleh
  representasi terpusat."

**T53. Langkah berikutnya? (Q44)**
- Data lapangan yang benar-benar rusak; jendela adaptif untuk deteksi lebih
  cepat; fitur level untuk bias; GroupShuffleSplit pada tahap sensor-ID.

**T54. Kalau ditanya angka yang tidak ada di layar? (Q45)**
- "Saya cek di `01_klaim_bukti.csv` atau notebook sumbernya." Jangan mengarang
  angka baru.

#### Bukti

- `Bahan_Sidang.md` Q44; `sidang/qa/06_mock_sidang.md` (latihan).

</details>